In [15]:
# =============================================================
# CELL 1: CONFIGURATION — Edit these values for your setup
# =============================================================

import os

# --- Paths (edit these) ---
PROJECT_DIR = '.'
ENVISION_INDEX_PATH = os.path.join(PROJECT_DIR, 'envision_index.json')
SYSTEM_PROMPT_PATH = os.path.join(PROJECT_DIR, 'rag_system_prompt.txt')

# List your .IFC test files here (add as many as you have)
IFC_FILES = [
    os.path.join(PROJECT_DIR, 'rag_ifc.ifc'),
    # os.path.join(PROJECT_DIR, 'test_project_2.ifc'),
    # os.path.join(PROJECT_DIR, 'test_project_3.ifc'),
]

# --- Model settings ---
MODEL_NAME = 'llama3.3:70b'
OLLAMA_URL = 'http://localhost:11434'
TEMPERATURE = 0.1          # Low = more deterministic
CONTEXT_WINDOW = 128000    # Llama 3.3 supports 128K
MAX_OUTPUT_TOKENS = 8192   # Max tokens per response
NUM_RUNS = 3               # Repeat each experiment N times

# --- RAG settings ---
EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
CHROMA_DB_PATH = os.path.join(PROJECT_DIR, 'chroma_db')
RETRIEVAL_TOP_K = 15       # Number of chunks to retrieve per query

# --- Thread limits (safety for shared server) ---
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'
os.environ['OPENBLAS_NUM_THREADS'] = '4'
os.environ['NUMEXPR_MAX_THREADS'] = '4'

# --- Results directory ---
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

print('\u2705 Configuration loaded.')
print(f'  Project dir:  {PROJECT_DIR}')
print(f'  Model:        {MODEL_NAME}')
print(f'  IFC files:    {len(IFC_FILES)}')
print(f'  Runs per exp: {NUM_RUNS}')


✅ Configuration loaded.
  Project dir:  .
  Model:        llama3.3:70b
  IFC files:    1
  Runs per exp: 3


In [16]:
# =============================================================
# CELL 2: HELPER FUNCTIONS
# =============================================================

import json, time, datetime, traceback, hashlib
import requests
import pandas as pd

# ---------- BIM EXTRACTION ----------
def extract_ifc_metadata(filepath):
    """Extract structured metadata from an .IFC file."""
    import ifcopenshell
    import ifcopenshell.util.element as util
    model = ifcopenshell.open(filepath)
    data = {'source_file': os.path.basename(filepath)}

    # Project metadata
    for p in model.by_type('IfcProject'):
        data['project_name'] = p.Name or 'Unknown'
        data['project_description'] = p.Description or 'N/A'

    # Site information
    for s in model.by_type('IfcSite'):
        data['site_name'] = s.Name or 'Unknown'
        data['site_description'] = s.Description or 'N/A'
        if s.RefLatitude: data['latitude'] = str(s.RefLatitude)
        if s.RefLongitude: data['longitude'] = str(s.RefLongitude)

    # Materials (RA1.x, CR1.1)
    data['materials'] = [m.Name for m in model.by_type('IfcMaterial')]

    # Element type counts
    counts = {}
    for elem in model.by_type('IfcProduct'):
        t = elem.is_a()
        counts[t] = counts.get(t, 0) + 1
    data['element_summary'] = counts

    # Energy devices (RA2.x, CR1.2)
    data['energy_devices'] = [
        {'type': e.is_a(), 'name': e.Name or 'unnamed'}
        for e in model.by_type('IfcEnergyConversionDevice')
    ]
    # Water systems (RA3.x)
    data['water_systems'] = [
        {'type': e.is_a(), 'name': e.Name or 'unnamed'}
        for e in model.by_type('IfcFlowSegment')
    ][:50]  # limit to avoid huge output

    # Lighting (QL1.5)
    data['lighting_fixtures'] = [
        {'type': e.is_a(), 'name': e.Name or 'unnamed'}
        for e in model.by_type('IfcFlowTerminal')
    ][:50]

    # Sustainability property sets
    sus_props = {}
    for elem in model.by_type('IfcProduct')[:100]:
        try:
            psets = util.get_psets(elem)
            for name, props in psets.items():
                if 'sustain' in name.lower() or 'green' in name.lower():
                    sus_props[name] = {k: str(v) for k, v in props.items()}
        except: pass
    data['sustainability_properties'] = sus_props

    return data


# ---------- OLLAMA API ----------
def call_ollama(system_prompt, user_message, timeout=3600):
    """Send a prompt to Ollama and return the response text + timing."""
    start = time.time()
    try:
        r = requests.post(
            f'{OLLAMA_URL}/api/generate',
            json={
                'model': MODEL_NAME,
                'system': system_prompt,
                'prompt': user_message,
                'stream': False,
                'options': {
                    'num_ctx': CONTEXT_WINDOW,
                    'temperature': TEMPERATURE,
                    'num_predict': MAX_OUTPUT_TOKENS,
                }
            },
            timeout=timeout
        )
        elapsed = time.time() - start
        result = r.json()
        return {
            'response': result.get('response', ''),
            'elapsed_seconds': round(elapsed, 2),
            'eval_count': result.get('eval_count', 0),
            'prompt_eval_count': result.get('prompt_eval_count', 0),
            'success': True
        }
    except Exception as e:
        elapsed = time.time() - start
        return {
            'response': f'ERROR: {str(e)}',
            'elapsed_seconds': round(elapsed, 2),
            'eval_count': 0, 'prompt_eval_count': 0,
            'success': False
        }


# ---------- ENVISION DATA LOADER ----------
def load_envision_by_category():
    """Load envision_index.json and group credits by category."""
    with open(ENVISION_INDEX_PATH) as f:
        data = json.load(f)
    categories = {}
    for cid, credit in data['credits'].items():
        cat = credit['sheet']
        if cat not in categories:
            categories[cat] = {}
        categories[cat][cid] = credit
    return categories, data['metadata']


# ---------- RESULT SAVING ----------
def save_result(result_dict, filepath):
    """Save a result dictionary to a JSON file."""
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, 'w') as f:
        json.dump(result_dict, f, indent=2, default=str)
    print(f'    Saved: {filepath}')


def timestamp():
    return datetime.datetime.now().strftime('%Y%m%d_%H%M%S')


print('\u2705 Helper functions loaded.')


✅ Helper functions loaded.


In [17]:
# =============================================================
# CELL 3: BUILD RAG INDEX (run once)
# =============================================================

import chromadb
from llama_index.core import Document, VectorStoreIndex, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.ollama import Ollama
from llama_index.core import Settings

# Configure global LlamaIndex settings
Settings.llm = Ollama(
    model=MODEL_NAME,
    request_timeout=3600,
    temperature=TEMPERATURE
)
Settings.embed_model = HuggingFaceEmbedding(
    model_name=EMBEDDING_MODEL
)

print('Building vector index from envision_index.json...')
print('(This takes 1-2 minutes on first run)\n')

# Load Envision data
with open(ENVISION_INDEX_PATH) as f:
    envision_raw = json.load(f)

# Create one Document per credit (59 documents total)
documents = []
for credit_id, credit in envision_raw['credits'].items():
    # Build a rich text representation of the credit
    text = f"""Credit: {credit['credit_id']} - {credit['credit_name']}
Category: {credit['sheet']} / {credit['category']}
Points: {credit['points_display']}
{credit['intent']}
{credit['metric']}
{credit['applicability_description']}

Questions:"""
    for q in credit.get('questions', []):
        text += f"\n  {q['letter']}: {q['text']}"

    if credit.get('lookup_criteria'):
        text += '\n\nLookup Criteria:'
        for lc in credit['lookup_criteria']:
            text += f"\n  {lc['criterion']}: {lc['description']}"

    tab = credit.get('tabulation', {})
    if tab:
        text += '\n\nLevel Guides:'
        for lvl in ['Improved', 'Enhanced', 'Superior', 'Conserving']:
            g = tab.get(f'LevelGuide_{lvl}', '')
            if g: text += f"\n  {lvl}: {g}"

    pts = credit.get('points', {})
    if pts:
        text += f"""\n\nPoints Rubric: No Level={pts.get('No_Level',0)}, """
        text += f"""Improved={pts.get('Improved',0)}, Enhanced={pts.get('Enhanced',0)}, """
        text += f"""Superior={pts.get('Superior',0)}, Conserving={pts.get('Conserving',0)}, """
        text += f"""Restorative={pts.get('Restorative','N/A')}"""

    doc = Document(
        text=text,
        metadata={
            'credit_id': credit_id,
            'credit_name': credit['credit_name'],
            'category': credit['sheet'],
            'subcategory': credit['category'],
            'max_points': pts.get('Total', 0),
            'has_lookup': bool(credit.get('lookup_criteria')),
        }
    )
    documents.append(doc)

print(f'Created {len(documents)} credit documents.')

# Initialize ChromaDB (persistent on disk)
db_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

# Delete existing collection if rebuilding
try: db_client.delete_collection('envision_credits')
except: pass

chroma_collection = db_client.create_collection('envision_credits')
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Build and persist the index
rag_index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    embed_model=Settings.embed_model,
    show_progress=True
)

print(f'\n\u2705 RAG index built and saved to {CHROMA_DB_PATH}')
print(f'   Collection size: {chroma_collection.count()} documents')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building vector index from envision_index.json...
(This takes 1-2 minutes on first run)

Created 59 credit documents.


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/59 [00:00<?, ?it/s]


✅ RAG index built and saved to ./chroma_db
   Collection size: 59 documents


In [18]:
# =============================================================
# CELL 4: WEEKEND RUNNER — Run both experiments unattended
# =============================================================

print('=' * 70)
print('  ENVISION EXPERIMENT RUNNER')
print(f'  Started: {datetime.datetime.now()}')
print(f'  Model:   {MODEL_NAME}')
print(f'  Files:   {len(IFC_FILES)} IFC file(s)')
print(f'  Runs:    {NUM_RUNS} per experiment')
print('=' * 70)

# Load system prompt
with open(SYSTEM_PROMPT_PATH) as f:
    system_prompt = f.read()

# Load Envision data grouped by category
categories, meta = load_envision_by_category()
category_names = list(categories.keys())
print(f'\nEnvision categories: {category_names}')
print(f'Total credits: {sum(len(v) for v in categories.values())}\n')

# Create the RAG query engine
rag_query_engine = rag_index.as_query_engine(
    similarity_top_k=RETRIEVAL_TOP_K,
    system_prompt=system_prompt
)

# Master log for timing summary
timing_log = []

# =========================================================
# MAIN LOOP: For each IFC file x each run x each experiment
# =========================================================
for file_idx, ifc_path in enumerate(IFC_FILES):

    file_label = os.path.splitext(os.path.basename(ifc_path))[0]
    print(f'\n{"=" * 70}')
    print(f'FILE {file_idx+1}/{len(IFC_FILES)}: {file_label}')
    print(f'{"=" * 70}')

    # --- Extract BIM metadata ---
    print('\n  Extracting BIM metadata...')
    try:
        bim_data = extract_ifc_metadata(ifc_path)
        bim_json = json.dumps(bim_data, indent=2, default=str)
        print(f'  \u2705 Extracted: {len(bim_data.get("element_summary", {}))} element types')
    except Exception as e:
        print(f'  \u274C BIM extraction failed: {e}')
        print(f'  Skipping this file.')
        continue

    # Save BIM metadata for reference
    save_result(bim_data, os.path.join(
        RESULTS_DIR, file_label, 'bim_metadata.json'
    ))

    # =====================================================
    # EXPERIMENT A: ZERO-SHOT
    # =====================================================
    for run_num in range(1, NUM_RUNS + 1):
        run_dir = os.path.join(
            RESULTS_DIR, file_label, 'zero_shot', f'run_{run_num}'
        )
        os.makedirs(run_dir, exist_ok=True)

        print(f'\n  --- ZERO-SHOT | Run {run_num}/{NUM_RUNS} ---')
        run_start = time.time()
        all_category_results = {}

        for cat_name, cat_credits in categories.items():
            print(f'    Category: {cat_name} ({len(cat_credits)} credits)...', end=' ')

            # Build the prompt with ONLY this category's credits
            cat_json = json.dumps(cat_credits, indent=2, default=str)
            user_msg = f"""## Envision Reference Data — {cat_name} Category

{cat_json}

## Project BIM Metadata

{bim_json}

## Task
Evaluate this project against all {cat_name} credits shown above.
For each credit, determine applicability, evaluate questions against the
BIM metadata, determine the achievable level, calculate points, and
identify documentation gaps. Return your assessment as JSON following
the credit_assessments format in your system instructions."""

            result = call_ollama(system_prompt, user_msg, timeout=3600)

            # Save individual category result
            cat_safe = cat_name.replace(' ', '_').lower()
            save_result(result, os.path.join(run_dir, f'{cat_safe}.json'))
            all_category_results[cat_name] = result

            status = '\u2705' if result['success'] else '\u274C'
            print(f'{status} ({result["elapsed_seconds"]}s)')

        run_elapsed = round(time.time() - run_start, 2)

        # Save combined run summary
        summary = {
            'experiment': 'zero_shot',
            'file': file_label,
            'run': run_num,
            'model': MODEL_NAME,
            'temperature': TEMPERATURE,
            'total_elapsed_seconds': run_elapsed,
            'categories': {
                k: {
                    'elapsed_seconds': v['elapsed_seconds'],
                    'output_tokens': v['eval_count'],
                    'input_tokens': v['prompt_eval_count'],
                    'success': v['success'],
                }
                for k, v in all_category_results.items()
            },
            'timestamp': timestamp()
        }
        save_result(summary, os.path.join(run_dir, '_summary.json'))

        timing_log.append({
            'experiment': 'zero_shot', 'file': file_label,
            'run': run_num, 'seconds': run_elapsed
        })
        print(f'  \u2705 Zero-shot run {run_num} complete: {run_elapsed}s total')

    # =====================================================
    # EXPERIMENT B: RAG
    # =====================================================
    for run_num in range(1, NUM_RUNS + 1):
        run_dir = os.path.join(
            RESULTS_DIR, file_label, 'rag', f'run_{run_num}'
        )
        os.makedirs(run_dir, exist_ok=True)

        print(f'\n  --- RAG | Run {run_num}/{NUM_RUNS} ---')
        run_start = time.time()
        all_category_results = {}

        for cat_name in category_names:
            n_credits = len(categories[cat_name])
            print(f'    Category: {cat_name} ({n_credits} credits)...', end=' ')

            query_text = f"""Evaluate this infrastructure project against all
Envision credits in the {cat_name} category.

Project BIM Metadata:
{bim_json}

For each credit in {cat_name}, determine applicability, evaluate questions,
determine the achievable level, calculate points, and identify
documentation gaps. Return JSON following the credit_assessments format."""

            rag_start = time.time()
            try:
                rag_response = rag_query_engine.query(query_text)
                rag_elapsed = round(time.time() - rag_start, 2)

                # Collect source nodes for analysis
                sources = []
                if hasattr(rag_response, 'source_nodes'):
                    for node in rag_response.source_nodes:
                        sources.append({
                            'credit_id': node.metadata.get('credit_id', '?'),
                            'score': round(node.score, 4) if node.score else None,
                            'category': node.metadata.get('category', '?'),
                        })

                result = {
                    'response': str(rag_response),
                    'elapsed_seconds': rag_elapsed,
                    'retrieved_sources': sources,
                    'num_sources': len(sources),
                    'success': True
                }
                status = '\u2705'
            except Exception as e:
                rag_elapsed = round(time.time() - rag_start, 2)
                result = {
                    'response': f'ERROR: {str(e)}',
                    'elapsed_seconds': rag_elapsed,
                    'retrieved_sources': [],
                    'num_sources': 0,
                    'success': False
                }
                status = '\u274C'

            cat_safe = cat_name.replace(' ', '_').lower()
            save_result(result, os.path.join(run_dir, f'{cat_safe}.json'))
            all_category_results[cat_name] = result
            print(f'{status} ({result["elapsed_seconds"]}s, {result["num_sources"]} sources)')

        run_elapsed = round(time.time() - run_start, 2)

        summary = {
            'experiment': 'rag',
            'file': file_label,
            'run': run_num,
            'model': MODEL_NAME,
            'temperature': TEMPERATURE,
            'retrieval_top_k': RETRIEVAL_TOP_K,
            'embedding_model': EMBEDDING_MODEL,
            'total_elapsed_seconds': run_elapsed,
            'categories': {
                k: {
                    'elapsed_seconds': v['elapsed_seconds'],
                    'num_sources': v['num_sources'],
                    'success': v['success'],
                }
                for k, v in all_category_results.items()
            },
            'timestamp': timestamp()
        }
        save_result(summary, os.path.join(run_dir, '_summary.json'))

        timing_log.append({
            'experiment': 'rag', 'file': file_label,
            'run': run_num, 'seconds': run_elapsed
        })
        print(f'  \u2705 RAG run {run_num} complete: {run_elapsed}s total')


# =========================================================
# FINAL SUMMARY
# =========================================================
print('\n' + '=' * 70)
print('  ALL EXPERIMENTS COMPLETE')
print(f'  Finished: {datetime.datetime.now()}')
print('=' * 70)

# Save timing log
timing_df = pd.DataFrame(timing_log)
timing_df.to_csv(os.path.join(RESULTS_DIR, 'timing_summary.csv'), index=False)
print('\nTiming summary:')
print(timing_df.to_string(index=False))

print(f'\n\u2705 All results saved to: {RESULTS_DIR}/')
print('\nFolder structure:')
print('  results/')
print('  \u251C\u2500\u2500 timing_summary.csv')
print('  \u251C\u2500\u2500 <project_name>/')
print('  \u2502   \u251C\u2500\u2500 bim_metadata.json')
print('  \u2502   \u251C\u2500\u2500 zero_shot/')
print('  \u2502   \u2502   \u251C\u2500\u2500 run_1/')
print('  \u2502   \u2502   \u2502   \u251C\u2500\u2500 _summary.json')
print('  \u2502   \u2502   \u2502   \u251C\u2500\u2500 quality_of_life.json')
print('  \u2502   \u2502   \u2502   \u251C\u2500\u2500 leadership.json')
print('  \u2502   \u2502   \u2502   \u2514\u2500\u2500 ...')
print('  \u2502   \u2502   \u251C\u2500\u2500 run_2/')
print('  \u2502   \u2502   \u2514\u2500\u2500 run_3/')
print('  \u2502   \u2514\u2500\u2500 rag/')
print('  \u2502       \u251C\u2500\u2500 run_1/')
print('  \u2502       \u2502   \u251C\u2500\u2500 _summary.json')
print('  \u2502       \u2502   \u251C\u2500\u2500 quality_of_life.json  (includes retrieved sources)')
print('  \u2502       \u2502   \u2514\u2500\u2500 ...')
print('  \u2502       \u251C\u2500\u2500 run_2/')
print('  \u2502       \u2514\u2500\u2500 run_3/')


  ENVISION EXPERIMENT RUNNER
  Started: 2026-04-18 10:54:02.279675
  Model:   llama3.3:70b
  Files:   1 IFC file(s)
  Runs:    3 per experiment

Envision categories: ['Climate And Resilience', 'Leadership', 'Natural World', 'Quality of Life', 'Resource Allocation']
Total credits: 59


FILE 1/1: rag_ifc

  Extracting BIM metadata...
  ✅ Extracted: 31 element types
    Saved: ./results/rag_ifc/bim_metadata.json

  --- ZERO-SHOT | Run 1/3 ---
    Category: Climate And Resilience (9 credits)...     Saved: ./results/rag_ifc/zero_shot/run_1/climate_and_resilience.json
✅ (3270.67s)
    Category: Leadership (11 credits)...     Saved: ./results/rag_ifc/zero_shot/run_1/leadership.json
❌ (3600.1s)
    Category: Natural World (13 credits)...     Saved: ./results/rag_ifc/zero_shot/run_1/natural_world.json
✅ (1636.09s)
    Category: Quality of Life (13 credits)...     Saved: ./results/rag_ifc/zero_shot/run_1/quality_of_life.json
✅ (1142.56s)
    Category: Resource Allocation (13 credits)...     Save